# Introduction to the Optical Eye Model: Anatomy, Materials, and Ray Tracing Theory

This notebook implements a **2D ray tracing simulation** of the human eye based on real optical data exported from Zemax. The goal is to understand how light propagates through the different structures of the eye – from the cornea to the retina – and to visualise the ray paths using Python.

Below we provide a self‑contained introduction to the anatomy, optical materials, and mathematical principles that underpin the code that follows.

## 1. Why model the eye optically?

A computational eye model allows us to study focusing, aberrations, and the effect of contact lenses or intraocular lenses without physical experiments. By rebuilding a Zemax model in Python, we gain complete control over every surface and can visualise the ray tracing step by step.

## 2. Basic anatomy from an optical perspective

Light enters the eye from the left (along the optical axis, labelled $z$) and passes through:

- **Cornea** – the transparent front surface. Radius of curvature ≈ $7.8$ mm, refractive index $n \approx 1.38$. It provides most of the eye’s focusing power.
- **Aqueous humor** – the fluid behind the cornea, $n \approx 1.34$.
- **Pupil** – the aperture stop of the eye, typically $1.25$ mm semi‑diameter in this model.
- **Crystalline lens** – a flexible lens with a **gradient refractive index (GRIN)**: the index is highest at the centre ($\approx 1.42$) and decreases towards the edges ($\approx 1.37$). This reduces spherical aberration.
- **Vitreous humor** – the gel‑like fluid filling the main cavity, $n \approx 1.34$.
- **Retina** – the image plane, often curved (radius $-12$ mm) where light is detected.

## 3. The Zemax data and its interpretation

The model is defined by a sequence of surfaces. Each surface has a radius of curvature $R$ (positive if the centre of curvature is to the left of the vertex, negative if to the right, infinite for a plane), a thickness (distance to the next surface), and refractive indices before and after the surface.

The table below translates the exported Zemax data into anatomical structures:

| Surface # | Anatomical part                | Radius (mm) | Thickness (mm) | Refractive index (n)     |
|-----------|--------------------------------|-------------|----------------|--------------------------|
| 0         | Object (at infinity)           | ∞           | $10^9$         | 1.0 (air)                |
| 1         | Dummy beam surface             | ∞           | 25.0           | 1.0 (air)                |
| 2         | Contact lens front surface     | 100.0       | 3.0            | 1.585 (polycarbonate)    |
| 3         | Contact lens back surface      | ∞ (plano)   | 25.0           | 1.0 (air)                |
| 4         | Cornea                         | 7.77        | 0.5            | 1.38                     |
| 5         | Aqueous humor                  | 6.40        | 3.16           | 1.34                     |
| 6         | Pupil (stop)                   | ∞           | 0.0            | (same as previous)       |
| 7         | Crystalline lens (front)       | 12.4        | 1.59           | GRIN (see below)         |
| 8         | Crystalline lens (back)        | ∞           | 2.43           | GRIN                     |
| 9         | Vitreous humor                 | -8.10       | 17.0           | 1.34                     |
| 10        | Retina (image plane)           | -12.0       | –              | –                        |

**Note:** The large $25$ mm air gaps are unusual for a real eye; they likely represent a laboratory setup. The code will reproduce the Zemax geometry exactly.

## 4. The Gradient‑Index (GRIN) crystalline lens

Unlike a homogeneous lens, the GRIN lens has a refractive index that varies with radial distance $r$ from the optical axis. In Zemax’s **Gradient 3** model, the index is given by a polynomial:

$$
n(r) = n_{00} + n_{01} r^2 + n_{02} r^4 + \cdots
$$

From the second data table, the coefficients for the front surface of the lens are:

- $n_{00} = 1.368$ (axial index)
- $n_{01} = -1.978 \times 10^{-3} \, \text{mm}^{-2}$ (quadratic term)
- $n_{02} = -0.016 \, \text{mm}^{-4}$ (quartic term)

Thus:

$$
n(r) = 1.368 - 0.001978\, r^2 - 0.016\, r^4
$$

The negative coefficients mean the index **decreases** away from the axis, which helps focus peripheral rays closer to the same point as paraxial rays – reducing spherical aberration.

The back surface of the lens has slightly different coefficients (axial index $1.407$), reflecting the natural gradient inside the lens.

## 5. Ray tracing theory: how the code works

The code traces each ray sequentially through all surfaces. For each step:

### 5.1 Intersection with a surface

The ray starts at a point $(z_{\text{ray}}, y_{\text{ray}})$ with direction angle $\theta$ measured from the $z$‑axis. Its path is described parametrically:

$$
\begin{aligned}
z(t) &= z_{\text{ray}} + t \cdot 1,\\
y(t) &= y_{\text{ray}} + t \cdot \tan\theta,
\end{aligned}
$$

with $t \ge 0$ (distance along $z$).

- **For a spherical surface** of radius $R$ whose vertex is at $z_s$, the centre is at $z_c = z_s - R$. Substituting into the circle equation $(z - z_c)^2 + y^2 = R^2$ gives a quadratic in $t$. The smallest positive root is the intersection point.
- **For a plane** ($R = \infty$), the intersection is simply $z = z_s$, and $y = y_{\text{ray}} + (z_s - z_{\text{ray}})\tan\theta$.

### 5.2 Refraction (Snell’s law)

At the intersection point, we compute the angle $\alpha$ of the surface normal relative to the $z$‑axis:

- Sphere: $\alpha = \arctan\left(\frac{y}{z - z_c}\right)$
- Plane: $\alpha = 0$

The incident angle relative to the normal is $\theta_i = \theta_{\text{ray}} - \alpha$. Snell’s law gives the transmitted angle:

$$
n_1 \sin\theta_i = n_2 \sin\theta_t \quad\Longrightarrow\quad \theta_t = \arcsin\!\left(\frac{n_1}{n_2}\sin\theta_i\right).
$$

The new ray direction is then:

$$
\theta_{\text{new}} = \alpha + \theta_t.
$$

If $\frac{n_1}{n_2}\sin\theta_i > 1$, total internal reflection occurs – the ray stops (though this rarely happens inside the eye).

### 5.3 Handling the GRIN lens (simplification in the first version)

A true GRIN would bend the ray continuously inside the lens. For simplicity, the first version of the code replaces the GRIN lens with a **homogeneous lens** whose index is the average of the front and back axial indices (approximately $1.3875$). This allows a working ray tracer with straight segments. A later extension can implement a multi‑layer slicing method for the full GRIN effect.

## 6. What the code does

The Python code that follows this markdown cell:

1. Defines each surface using the data above (positions, radii, indices).
2. Implements functions for sphere/plane intersection and Snell’s law.
3. Traces a set of parallel rays (from infinity) at different initial heights $y_0$ (e.g., from $-3$ mm to $+3$ mm).
4. Plots the 2D ray paths on a $z$‑$y$ plane, overlaying the surfaces as vertical lines.
5. Outputs a visualisation of how the eye model focuses light.

## 7. Possible extensions

- Implement the full GRIN lens by slicing it into many thin layers.
- Apply the pupil stop (surface 6) to block rays whose height exceeds the semi‑diameter $1.25$ mm.
- Compute the spot diagram on the retina.
- Adjust thicknesses to match a realistic human eye (e.g., shorter air gaps).

